In [12]:

# %%
# 0) Imports and environment setup

import os
import sys
import math
import random
import logging
import warnings
import yaml
import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)


Using device: cpu



## 1) Paths and config selection

These mirror the way your MTAD-GAT notebook loads YAML configs.

- `EXP_ID` is typically one of the experiment keys in `mtad_gat.yaml` (e.g., `mtad_gat_SMD`).
- `DATASET_CFG_FILE` is your dataset YAML (e.g., `SMD.yaml`).

If you want to run a different dataset (WADI, SWaT, etc.), use the corresponding YAML you already have.


In [14]:

# %%
# 1) Config selection (edit these)

EXP_ID = "mtad_gat_SMD"
DATASET_CFG_FILE = "SMD.yaml"

# Path to MTGFlow repo (expected to be next to this notebook)
MTGFLOW_REPO_PATH = "./MTGFLOW-main"   # <-- change if needed

# For SMD, mtad-tools often loads a list of entities (machines).
# MTGFlow is multivariate; we typically train per-entity (machine) treating features as sensors.
ENTITY_INDEX = 0  # choose which entity to run, e.g. 0 for machine-1, 1 for machine-2, etc.

# Training hyperparameters for MTGFlow (you can change freely)
MTGFLOW_EPOCHS = 40
MTGFLOW_LR = 2e-3
MTGFLOW_WEIGHT_DECAY = 5e-4
MTGFLOW_BATCH_SIZE = 256
GRAD_CLIP = 1.0

print("EXP_ID:", EXP_ID)
print("DATASET_CFG_FILE:", DATASET_CFG_FILE)
print("MTGFLOW_REPO_PATH:", MTGFLOW_REPO_PATH)


EXP_ID: mtad_gat_SMD
DATASET_CFG_FILE: SMD.yaml
MTGFLOW_REPO_PATH: ./MTGFLOW-main



## 2) Load YAML configs (same pattern as MTAD-GAT)

This cell reads:
- model config from `mtad_gat.yaml` (to reuse `window_size`, `stride`, etc.)
- dataset config from `SMD.yaml` (to reuse `data_root`, `entities`, etc.)


In [15]:

# %%
def load_model_config(model_cfg_path="mtad_gat.yaml", exp_id="mtad_gat_SMD"):
    with open(model_cfg_path, "r") as f:
        cfg_all = yaml.safe_load(f)

    if "Base" in cfg_all:
        base = cfg_all["Base"] or {}
    else:
        base = {}

    if exp_id not in cfg_all:
        raise ValueError(f"Experiment id '{exp_id}' not found. Available: {list(cfg_all.keys())}")

    merged = dict(base)
    merged.update(cfg_all[exp_id] or {})
    return merged

def load_dataset_config(dataset_cfg_path="SMD.yaml"):
    with open(dataset_cfg_path, "r") as f:
        return yaml.safe_load(f)

model_params = load_model_config("mtad_gat.yaml", EXP_ID)
dataset_params = load_dataset_config(DATASET_CFG_FILE)

print("=== Model params (subset) ===")
for k in ["dataset_id", "batch_size", "window_size", "stride", "nb_epoch"]:
    if k in model_params:
        print(f"{k}: {model_params[k]}")

print("\n=== Dataset params (subset) ===")
for k in ["dataset", "data_root", "dim", "entities"]:
    if k in dataset_params:
        v = dataset_params[k]
        if k == "entities" and isinstance(v, list):
            print(f"{k}: list(len={len(v)}) first5={v[:5]}")
        else:
            print(f"{k}: {v}")


=== Model params (subset) ===
dataset_id: SMD
batch_size: 512
window_size: 100
stride: 1
nb_epoch: 10

=== Dataset params (subset) ===



## 3) Load data and generate windows (mtad-tools)

Stage A (Step 3): Build windows cache per entity
Input: raw SMD files
Output per entity: cache_windows_smd/{entity}_T{T}_S{S}.npz

train_windows [N,T,K]
test_windows [N,T,K]
test_labels_window [N]


In [37]:
# %% [STEP 3 - Multi-dataset] Cache windows per entity to disk (with starts)

import os, gc, time
import numpy as np
from dataloader import load_dataset
from data_preprocess import generate_windows

DATASET_NAME = str(dataset_params.get("dataset", "UNKNOWN_DATASET"))
window_size = int(model_params.get("window_size", 100))
stride = int(model_params.get("stride", 1))

CACHE_DIR = os.path.join("./cache_windows", DATASET_NAME, f"T{window_size}_S{stride}")
os.makedirs(CACHE_DIR, exist_ok=True)

all_entities = dataset_params["entities"]

print("Dataset:", DATASET_NAME)
print("Caching to:", os.path.abspath(CACHE_DIR))
print("window_size:", window_size, "| stride:", stride, "| num_entities:", len(all_entities))

for i, entity in enumerate(all_entities):
    cache_path = os.path.join(CACHE_DIR, f"{entity}.npz")

    if os.path.exists(cache_path):
        print(f"[{i+1:02d}/{len(all_entities)}] {entity}: already cached -> skip")
        continue

    print(f"\n[{i+1:02d}/{len(all_entities)}] Caching entity={entity}")
    t0 = time.time()

    # Load ONE entity (normalization happens inside load_dataset)
    data_dict_1 = load_dataset(
        data_root=data_root,
        entities=[entity],
        valid_ratio=dataset_params.get("valid_ratio", 0.0),
        dim=int(dataset_params["dim"]),
        test_label_postfix=dataset_params["test_label_postfix"],
        test_postfix=dataset_params["test_postfix"],
        train_postfix=dataset_params["train_postfix"],
    )

    # Windowing for this entity
    window_dict_1 = generate_windows(
        data_dict_1,
        window_size=window_size,
        stride=stride,
    )

    w = window_dict_1[entity]
    train_windows = w["train_windows"].astype(np.float32)        # [N_train, T, K]
    test_windows  = w["test_windows"].astype(np.float32)         # [N_test,  T, K]
    test_labels_point = w["test_label"].astype(np.float32)       # [N_test,  T]

    test_labels_window = (test_labels_point.sum(axis=1) > 0).astype(np.int64)

    train_starts = (np.arange(train_windows.shape[0], dtype=np.int64) * stride)
    test_starts  = (np.arange(test_windows.shape[0],  dtype=np.int64) * stride)

    np.savez_compressed(
        cache_path,
        train_windows=train_windows,
        test_windows=test_windows,
        test_labels_window=test_labels_window,
        train_starts=train_starts,
        test_starts=test_starts,
        meta=np.array([DATASET_NAME, entity, window_size, stride], dtype=object),
    )

    anomaly_ratio = float(test_labels_window.mean()) if test_labels_window.size > 0 else 0.0
    print("  saved:", cache_path)
    print("  train:", train_windows.shape, "| test:", test_windows.shape,
          "| anomaly_ratio:", anomaly_ratio,
          "| elapsed_sec:", round(time.time() - t0, 2))

    # cleanup
    del data_dict_1, window_dict_1, w
    del train_windows, test_windows, test_labels_point, test_labels_window, train_starts, test_starts
    gc.collect()

print("\n✅ Window caching finished.")

2026-03-03 15:01:14,377 - INFO - Loading data from ../data/SMD/
2026-03-03 15:01:14,513 - INFO - Loading 1 entities done.
2026-03-03 15:01:14,516 - INFO - Train/Valid/Test: 28479/0/28479 lines.
2026-03-03 15:01:14,517 - INFO - Generating sliding windows (size 100).


Dataset: smd
Caching to: C:\Users\S4027872\OneDrive - RMIT University\Desktop\My-Study\Anomaly detection\code\mtad-gat-independent\cache_windows\smd\T100_S1
window_size: 100 | stride: 1 | num_entities: 28

[01/28] Caching entity=machine-1-1


2026-03-03 15:01:14,941 - INFO - Windows for train #: (28379, 100, 38)
2026-03-03 15:01:15,586 - INFO - Windows for test #: (28379, 100, 38)


  saved: ./cache_windows\smd\T100_S1\machine-1-1.npz
  train: (28379, 100, 38) | test: (28379, 100, 38) | anomaly_ratio: 0.12283730927798725 | elapsed_sec: 11.34


2026-03-03 15:01:26,691 - INFO - Loading data from ../data/SMD/
2026-03-03 15:01:26,815 - INFO - Loading 1 entities done.
2026-03-03 15:01:26,818 - INFO - Train/Valid/Test: 23694/0/23694 lines.
2026-03-03 15:01:26,820 - INFO - Generating sliding windows (size 100).


...

  saved: ./cache_windows\smd\T100_S1\machine-3-10.npz
  train: (23592, 100, 38) | test: (23593, 100, 38) | anomaly_ratio: 0.09892764803119569 | elapsed_sec: 8.03


2026-03-03 15:05:29,624 - INFO - Loading data from ../data/SMD/
2026-03-03 15:05:29,703 - INFO - Loading 1 entities done.
2026-03-03 15:05:29,703 - INFO - Train/Valid/Test: 28695/0/28696 lines.
2026-03-03 15:05:29,703 - INFO - Generating sliding windows (size 100).



[28/28] Caching entity=machine-3-11


2026-03-03 15:05:29,894 - INFO - Windows for train #: (28595, 100, 38)
2026-03-03 15:05:30,218 - INFO - Windows for test #: (28595, 100, 38)


  saved: ./cache_windows\smd\T100_S1\machine-3-11.npz
  train: (28595, 100, 38) | test: (28596, 100, 38) | anomaly_ratio: 0.015386767380053155 | elapsed_sec: 10.64

✅ Window caching finished.


Raw time-series
      ↓
Step 3 → Window cache (disk)
      ↓
Step 4 → DataLoader (reshape only)
      ↓
Step 5 → Build MTGFlow model
      ↓
Step 6 → Train model (maximize likelihood)
      ↓
Evaluation → Compute anomaly scores + AUROC

##Stage B (Steps 4–6): Train/eval per entity using cached windows

Read .npz
Create Dataset/DataLoader (Step 4)
Build model (Step 5)
Train/eval (Step 6)

Save:
models/{entity}.pt
scores/{entity}.npz (scores + labels)
append metrics to results.csv/jsonl

In [40]:
# =========================
# MTGFlow: Shared Setup + Helpers (use once)
# =========================

import os, json, gc, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score
from torch.nn.utils import clip_grad_value_

# ---- device ----
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ---- dataset/cache config ----
DATASET_NAME = str(dataset_params.get("dataset", "UNKNOWN_DATASET"))
window_size = int(model_params.get("window_size", 100))
stride = int(model_params.get("stride", 1))

CACHE_DIR = os.path.join("./cache_windows", DATASET_NAME, f"T{window_size}_S{stride}")
assert os.path.isdir(CACHE_DIR), f"Cache dir not found: {CACHE_DIR}"
print("Cache dir:", os.path.abspath(CACHE_DIR))

# ---- training hyperparams (override if you want) ----
MTGFLOW_BATCH_SIZE = int(model_params.get("batch_size", 512))
MTGFLOW_EPOCHS = int(model_params.get("nb_epoch", 10))  # set in YAML: nb_epoch
MTGFLOW_LR = 1e-3
MTGFLOW_WEIGHT_DECAY = 0.0
GRAD_CLIP = 1.0

# ---- model repo import ----
if MTGFLOW_REPO_PATH not in os.sys.path:
    os.sys.path.insert(0, MTGFLOW_REPO_PATH)
from models.MTGFLOW import MTGFLOW

def load_cached_entity(entity, cache_dir=CACHE_DIR):
    path = os.path.join(cache_dir, f"{entity}.npz")
    d = np.load(path, allow_pickle=True)
    return {
        "path": path,
        "train_windows": d["train_windows"],              # [N,T,K]
        "test_windows": d["test_windows"],                # [N,T,K]
        "test_labels_window": d["test_labels_window"],    # [N]
        "train_starts": d["train_starts"],                # [N]
        "test_starts": d["test_starts"],                  # [N]
        "meta": d["meta"] if "meta" in d.files else None,
    }

class MTGFlowWindowDataset(Dataset):
    def __init__(self, windows, window_labels=None, starts=None):
        self.windows = windows.astype(np.float32)  # [N,T,K]
        self.labels = None if window_labels is None else window_labels.astype(np.int64)
        self.starts = None if starts is None else starts.astype(np.int64)

    def __len__(self):
        return self.windows.shape[0]

    def __getitem__(self, idx):
        w = self.windows[idx]                      # [T,K]
        x = torch.from_numpy(w).transpose(0, 1)    # [K,T]
        x = x.unsqueeze(-1)                        # [K,T,1]
        y = 0 if self.labels is None else int(self.labels[idx])
        s = -1 if self.starts is None else int(self.starts[idx])
        return x, y, int(idx), s

def make_loaders(train_windows, test_windows, test_labels_window, train_starts, test_starts):
    train_ds = MTGFlowWindowDataset(train_windows, None, train_starts)
    test_ds  = MTGFlowWindowDataset(test_windows, test_labels_window, test_starts)

    train_loader = DataLoader(train_ds, batch_size=MTGFLOW_BATCH_SIZE, shuffle=True, drop_last=True)
    test_loader  = DataLoader(test_ds,  batch_size=MTGFLOW_BATCH_SIZE, shuffle=False, drop_last=False)
    return train_loader, test_loader, train_ds, test_ds

def evaluate_scores(model, loader):
    model.eval()
    scores, labels, starts, indices = [], [], [], []
    with torch.no_grad():
        for x, y, idx, s in loader:
            x = x.to(DEVICE)
            # model.test(x) returns log-likelihood per sample; score = NLL = -loglik
            nll = (-model.test(x)).detach().cpu().numpy().reshape(-1)
            scores.append(nll)
            labels.append(np.asarray(y, dtype=np.int64).reshape(-1))
            starts.append(np.asarray(s, dtype=np.int64).reshape(-1))
            indices.append(np.asarray(idx, dtype=np.int64).reshape(-1))

    scores = np.concatenate(scores) if len(scores) else np.array([])
    labels = np.concatenate(labels) if len(labels) else np.array([])
    starts = np.concatenate(starts) if len(starts) else np.array([])
    indices = np.concatenate(indices) if len(indices) else np.array([])

    if len(labels) > 0 and len(np.unique(labels)) > 1:
        auroc = float(roc_auc_score(labels, scores))
        aupr  = float(average_precision_score(labels, scores))
    else:
        auroc, aupr = float("nan"), float("nan")

    return auroc, aupr, scores, labels, indices, starts

def build_mtgflow_model(K, T, n_blocks=2, hidden_size=32, n_hidden=1, dropout=0.0, model_type="MAF", batch_norm=False):
    model = MTGFLOW(
        n_blocks=n_blocks,
        input_size=1,
        hidden_size=hidden_size,
        n_hidden=n_hidden,
        window_size=T,
        n_sensor=K,
        dropout=dropout,
        model=model_type,
        batch_norm=batch_norm,
    ).to(DEVICE)
    return model

def train_one_entity_mtgflow(entity, pack, results_dir, n_blocks=2):
    """
    Trains MTGFlow for one entity.
    Saves:
      - per-epoch history CSV
      - best model checkpoint (.pt)
      - final test scores mapping (.npz)
    Returns a dict of summary metrics.
    """
    os.makedirs(results_dir, exist_ok=True)
    models_dir = os.path.join(results_dir, "models")
    scores_dir = os.path.join(results_dir, "scores")
    history_dir = os.path.join(results_dir, "history")
    os.makedirs(models_dir, exist_ok=True)
    os.makedirs(scores_dir, exist_ok=True)
    os.makedirs(history_dir, exist_ok=True)

    train_windows = pack["train_windows"]
    test_windows  = pack["test_windows"]
    test_labels_window = pack["test_labels_window"]
    train_starts = pack["train_starts"]
    test_starts  = pack["test_starts"]

    K = int(train_windows.shape[-1])
    T = int(train_windows.shape[1])
    anomaly_ratio = float(test_labels_window.mean()) if len(test_labels_window) else 0.0

    train_loader, test_loader, _, _ = make_loaders(train_windows, test_windows, test_labels_window, train_starts, test_starts)

    model = build_mtgflow_model(K, T, n_blocks=n_blocks)
    optimizer = torch.optim.Adam(
        [{"params": model.parameters(), "weight_decay": MTGFLOW_WEIGHT_DECAY}],
        lr=MTGFLOW_LR,
        weight_decay=0.0,
    )

    history = {"epoch": [], "train_nll": [], "test_auroc": [], "test_aupr": []}
    best_auroc, best_state = -1.0, None

    for epoch in range(MTGFLOW_EPOCHS):
        model.train()
        losses = []

        for x, _, _, _ in train_loader:
            x = x.to(DEVICE)
            optimizer.zero_grad()
            loss = -model(x)  # maximize log-likelihood
            loss.backward()
            clip_grad_value_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            losses.append(float(loss.item()))

        train_nll = float(np.mean(losses)) if len(losses) else float("nan")
        auroc, aupr, _, _, _, _ = evaluate_scores(model, test_loader)

        history["epoch"].append(int(epoch + 1))
        history["train_nll"].append(train_nll)
        history["test_auroc"].append(float(auroc))
        history["test_aupr"].append(float(aupr))

        if np.isfinite(auroc) and auroc > best_auroc:
            best_auroc = float(auroc)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # restore best
    if best_state is not None:
        model.load_state_dict(best_state)

    # final eval + save scores
    auroc, aupr, scores, labels, indices, starts = evaluate_scores(model, test_loader)

    model_path = os.path.join(models_dir, f"{entity}.pt")
    torch.save(
        {
            "state_dict": model.state_dict(),
            "entity": entity,
            "dataset": DATASET_NAME,
            "K": K,
            "T": T,
            "window_size": window_size,
            "stride": stride,
            "n_blocks": n_blocks,
        },
        model_path,
    )

    scores_path = os.path.join(scores_dir, f"{entity}.npz")
    np.savez_compressed(scores_path, scores=scores, labels=labels, indices=indices, starts=starts)

    history_path = os.path.join(history_dir, f"{entity}_history.csv")
    pd.DataFrame(history).to_csv(history_path, index=False)

    # cleanup
    del model, optimizer
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "dataset": DATASET_NAME,
        "entity": entity,
        "K": K,
        "T": T,
        "n_train_windows": int(train_windows.shape[0]),
        "n_test_windows": int(test_windows.shape[0]),
        "anomaly_ratio": float(anomaly_ratio),
        "auroc": float(auroc),
        "aupr": float(aupr),
        "best_auroc_during_training": float(best_auroc) if best_state is not None else float("nan"),
        "model_path": model_path,
        "scores_path": scores_path,
        "history_path": history_path,
    }

Using device: cpu
Cache dir: C:\Users\S4027872\OneDrive - RMIT University\Desktop\My-Study\Anomaly detection\code\mtad-gat-independent\cache_windows\smd\T100_S1


## A) Learning mode — train ONE entity interactively

In [ ]:
# =========================
# A) Interactive (Learning Mode): Train ONE entity + save per-epoch history
# =========================

# Pick one entity
ENTITY_INDEX = 0
entity = dataset_params["entities"][ENTITY_INDEX]

pack = load_cached_entity(entity)
print("Entity:", entity)
print("train_windows:", pack["train_windows"].shape, "test_windows:", pack["test_windows"].shape,
      "anomaly_ratio:", float(pack["test_labels_window"].mean()))

# Results directory for interactive runs
INTERACTIVE_DIR = os.path.join("./results_mtgflow_interactive", DATASET_NAME, f"T{window_size}_S{stride}", entity)
os.makedirs(INTERACTIVE_DIR, exist_ok=True)

# Train and save artifacts
summary = train_one_entity_mtgflow(entity, pack, results_dir=INTERACTIVE_DIR, n_blocks=2)
print("\nSaved:")
print(" - history:", summary["history_path"])
print(" - model  :", summary["model_path"])
print(" - scores :", summary["scores_path"])
print("\nFinal metrics:")
print(" AUROC:", summary["auroc"])
print(" AUPR :", summary["aupr"])

# Show Top-K anomalous windows with time mapping (uses saved scores)
d = np.load(summary["scores_path"])
scores = d["scores"]; labels = d["labels"]; indices = d["indices"]; starts = d["starts"]
T = summary["T"]

top_k = 5
top = np.argsort(-scores)[:top_k]
print("\nTop anomalous windows:")
for r, j in enumerate(top, 1):
    start = int(starts[j])
    end = start + T - 1
    print(f"#{r}: test_window_idx={int(indices[j])} | t=[{start},{end}] | score={float(scores[j]):.3f} | label={int(labels[j])}")

Entity: machine-1-1
train_windows: (28379, 100, 38) test_windows: (28379, 100, 38) anomaly_ratio: 0.12283730927798725


In [ ]:
# =========================
# B) Experiment Mode: Loop ALL entities sequentially + save per-epoch history per entity
# =========================

RESULTS_DIR = os.path.join("./results_mtgflow", DATASET_NAME, f"T{window_size}_S{stride}")
os.makedirs(RESULTS_DIR, exist_ok=True)

RESULTS_CSV = os.path.join(RESULTS_DIR, "per_entity_results.csv")
RESULTS_JSONL = os.path.join(RESULTS_DIR, "per_entity_results.jsonl")

def append_jsonl(path, obj):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj) + "\n")

rows = []

entities = dataset_params["entities"]
for i, entity in enumerate(entities):
    print(f"\n==================== [{i+1}/{len(entities)}] entity={entity} ====================")
    t0 = time.time()

    pack = load_cached_entity(entity)
    summary = train_one_entity_mtgflow(entity, pack, results_dir=RESULTS_DIR, n_blocks=2)
    summary["elapsed_sec"] = float(time.time() - t0)

    append_jsonl(RESULTS_JSONL, summary)
    rows.append(summary)
    pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)

    print(f"Done {entity} | AUROC={summary['auroc']:.4f} | AUPR={summary['aupr']:.4f} | history saved")

    # free pack arrays (np.load already loaded into RAM)
    del pack
    gc.collect()

print("\n✅ Finished all entities.")
print("Results CSV :", RESULTS_CSV)
print("Results JSONL:", RESULTS_JSONL)
print("History dir :", os.path.join(RESULTS_DIR, "history"))
print("Models dir  :", os.path.join(RESULTS_DIR, "models"))
print("Scores dir  :", os.path.join(RESULTS_DIR, "scores"))